# The Allegheny Family Screening Tool: When Being Poor Looks Like Being Risky

In this notebook I look at the Allegheny Family Screening Tool, a child welfare risk-prediction system, as a twelfth mechanism behind an AI ethics failure: a model that can only see a family's involvement with public systems, deployed on a population where being poor is exactly what determines whether that involvement is public at all.

In this notebook, I will:

- Summarize what the tool does and the core criticism child welfare researchers and journalists have raised about it
- Simulate two income groups with an identical real underlying rate of a risk-relevant issue, differing only in which systems, public or private, they resolve it through
- Measure how that one difference in visibility produces a huge gap in both false alarms and missed real risk
- Try the fix critics specifically called for, removing the clearest poverty-only variable, and see that it closes one gap while leaving the other almost untouched
- Sit with why this particular mechanism doesn't have a clean technical fix, only mitigations

## 1. Background: What Was Reported

Starting in August 2016, Allegheny County, Pennsylvania, began using a predictive risk model, the Allegheny Family Screening Tool, to help call screeners at its child welfare hotline decide whether a report of suspected child maltreatment should be investigated in person. The tool generates a numeric risk score for each report, built from the county's integrated data warehouse, a system that links records from child welfare history, public behavioral health and substance use treatment, county jail bookings, and public benefits programs like food and cash assistance.

A central criticism, raised by researcher Virginia Eubanks in her book Automating Inequality and later echoed by journalists and academic reviewers who examined the tool, is about what that data warehouse can and cannot see. It only contains records generated by contact with public systems. A family that gets mental health care, substance use treatment, or other support through private insurance and private providers leaves no trace in it at all, regardless of what's actually happening in that family. A family that relies on public services for the same underlying needs, often because they have no other option, shows up in exactly the records the model is built from. The county has defended the tool, pointed out that it does not use race as an input, and revised it across versions, removing some variables and keeping a human screener in the loop with the authority to override the score. Independent evaluators and journalists examining the tool, including later investigations by the Associated Press and other newsrooms, have continued to find that families from lower-income and predominantly Black neighborhoods are flagged for investigation at disproportionately high rates relative to their actual outcomes.

## 2. Why This Is a Different Mechanism

This isn't a proxy feature quietly standing in for a protected attribute inside an otherwise representative dataset, and it isn't an explicit attribute typed directly into a score. It's a gap in what the data can see at all, one that lines up with income for a structural reason rather than a modeling choice: a family's economic situation is exactly what decides whether getting help for a given problem happens through a channel the county's data warehouse records, or through one it has no way to observe.

The same underlying issue, say a parent's substance use or a mental health crisis, can occur at the same rate in a wealthy household and a poor one. If the wealthy household resolves it privately, nothing about it ever reaches this system. If the poor household resolves it publicly, because that's the only option available to them, it becomes part of a risk score. The tool doesn't need to know anyone's income to produce this gap. Income decides which channel a family uses, and the model only has eyes on one of those channels.

## 3. A Note on the Simulation Below

This notebook does not use Allegheny County's real model, real variables, or real score thresholds, none of which are something I should try to reproduce. I use a small set of simplified attributes, whether a family has an underlying issue, whether it gets resolved publicly or privately, and whether the family separately receives a general public benefit unrelated to that issue, to make one mechanism concrete: how a model built only from public-system records can end up scoring poverty itself as risk, while remaining nearly blind to the same real issue in a wealthier household.

## 4. Simulating Two Income Groups With the Same True Rate of a Real Issue

I give both groups the exact same underlying rate of a risk-relevant issue, since conditions like substance use and mental health struggles occur across income levels at broadly similar rates in practice. The groups differ only in two ways that reflect how income actually shapes which systems a family relies on: how often a real issue gets resolved through a publicly recorded channel rather than a private one, and how often the family separately receives a general public benefit that has nothing to do with the issue at all.

In [ ]:
import random

random.seed(17)

N_PER_GROUP = 5000
TRUE_ISSUE_RATE = 0.15

GROUPS = [
    {"name": "Lower-income families", "resolution_public_rate": 0.90, "benefit_baseline_rate": 0.70},
    {"name": "Higher-income families", "resolution_public_rate": 0.05, "benefit_baseline_rate": 0.01},
]


def make_families(n, true_issue_rate, resolution_public_rate, benefit_baseline_rate):
    """Returns a list of family dicts with a true issue flag and two separate public-record flags."""
    families = []
    for _ in range(n):
        has_issue = random.random() < true_issue_rate
        resolved_publicly = has_issue and random.random() < resolution_public_rate
        receives_public_benefit = random.random() < benefit_baseline_rate
        families.append({
            "has_issue": has_issue,
            "resolved_publicly": resolved_publicly,
            "receives_public_benefit": receives_public_benefit,
        })
    return families

## 5. Building a Risk Score From Public Records Alone

The score below counts only what shows up in a public record: whether the family's issue, if they have one, was resolved through a publicly visible channel, and whether the family separately receives a general public benefit. Nothing here ever looks at income directly, and nothing here can see a privately resolved issue at all.